# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az")

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Customer ID cleanup

In [0]:
df = (
    df
    .withColumn(
        "customer_number",
        F.when(col("customer_number").startswith("NAS"), F.substring(col("customer_number"), 4, F.length(col("customer_number"))))
         .otherwise(col("customer_number"))
    )
)

## Birthdate validation

In [0]:
df = (
    df
    .withColumn(
        "birth_date",
        F.when(col("birth_date") > F.current_date(), None)
         .otherwise(col("birth_date"))
    )
)

## Gender normalization

In [0]:
df = (
    df
    .withColumn(
        "gender",
        F.when(F.upper(col("gender")).isin("F", "FEMALE"), "Female")
         .when(F.upper(col("gender")).isin("M", "MALE"), "Male")
         .otherwise("n/a")
    )
)

## Sanity check of final DataFrame

In [0]:
df.limit(10).display()

# Write into silver table

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("workspace.silver.erp_customers")
)